# 02 · LoRA fine-tune NLLB-600M on Sindhi–English (optional stretch)

Fine-tunes `facebook/nllb-200-distilled-600M` with LoRA on public parallel data (OPUS-100 en–sd,
plus anything you add), **never** on FLORES (that's the test set). Keep the result only if chrF++ on
FLORES devtest beats notebook 01. Runtime: ~1–2 h on a T4 for 1 epoch.

In [ ]:
# Session options (right panel): Accelerator = GPU T4 x2 (or P100), Internet = On.
!pip install -q -U transformers==4.57.1 accelerate sentencepiece sacrebleu ctranslate2==4.6.0 peft datasets
import torch, os
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# FLORES-200 devtest (1012 sentences) -- public download, no login needed
!wget -q -nc https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && tar xzf flores200_dataset.tar.gz
def read(path):
    with open(path, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f]
flores = {"sd": read("flores200_dataset/devtest/snd_Arab.devtest"),
          "en": read("flores200_dataset/devtest/eng_Latn.devtest")}
CODES = {"sd": "snd_Arab", "en": "eng_Latn"}
print(len(flores["sd"]), flores["sd"][0][:80])

In [ ]:
import sacrebleu, time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def translate(model, tok, sentences, src, tgt, batch_size=32, beams=4):
    tok.src_lang = CODES[src]
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = tok(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            gen = model.generate(**batch, forced_bos_token_id=tok.convert_tokens_to_ids(CODES[tgt]),
                                 num_beams=beams, max_new_tokens=256)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

def score(model, tok, n=1012):
    res = {}
    for src, tgt in (("sd", "en"), ("en", "sd")):
        t = time.time()
        hyps = translate(model, tok, flores[src][:n], src, tgt)
        refs = flores[tgt][:n]
        res[f"{src}-{tgt}"] = {"chrf++": round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 1),
                               "bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 1),
                               "sec": round(time.time() - t)}
    return res

In [ ]:
from datasets import load_dataset, concatenate_datasets
ds = load_dataset("Helsinki-NLP/opus-100", "en-sd")      # check the Hub if this pair moves
train = ds["train"]
print(train)

def clean(ex):
    en, sd = ex["translation"]["en"].strip(), ex["translation"]["sd"].strip()
    ratio = (len(en) + 1) / (len(sd) + 1)
    return 3 <= len(sd) <= 300 and 0.4 <= ratio <= 2.5 and any("؀" <= c <= "ۿ" for c in sd)

train = train.filter(clean)
flores_set = set(flores["sd"]) | set(flores["en"])
train = train.filter(lambda ex: ex["translation"]["sd"] not in flores_set and ex["translation"]["en"] not in flores_set)
print("after cleaning:", len(train))

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

BASE = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE)
base_scores = score(model.half().cuda().eval(), tok, n=300); print("before:", base_scores)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
                                         task_type="SEQ_2_SEQ_LM"))
model.print_trainable_parameters()

def encode(batch, src="sd", tgt="en"):
    # train both directions: half the rows sd->en, half en->sd
    tok.src_lang, tok.tgt_lang = CODES[src], CODES[tgt]
    s = [t[src] for t in batch["translation"]]; t_ = [t[tgt] for t in batch["translation"]]
    return tok(s, text_target=t_, max_length=200, truncation=True)

half = len(train) // 2
sd_en = train.select(range(half)).map(lambda b: encode(b, "sd", "en"), batched=True, remove_columns=train.column_names)
en_sd = train.select(range(half, len(train))).map(lambda b: encode(b, "en", "sd"), batched=True, remove_columns=train.column_names)
tokenized = concatenate_datasets([sd_en, en_sd]).shuffle(seed=0)

args = Seq2SeqTrainingArguments("nllb-lora", per_device_train_batch_size=16, gradient_accumulation_steps=2,
                                learning_rate=2e-4, num_train_epochs=1, warmup_ratio=0.03, fp16=True,
                                logging_steps=50, save_steps=1000, save_total_limit=2, report_to="none")
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tokenized,
                         data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()

In [ ]:
merged = model.merge_and_unload().half().cuda().eval()
after = score(merged, tok, n=300)
print("before:", base_scores)
print("after: ", after)

In [ ]:
# Ship it only if it's better on BOTH directions
if all(after[k]["chrf++"] > base_scores[k]["chrf++"] for k in after):
    merged.float().save_pretrained("nllb-sd-lora-merged"); tok.save_pretrained("nllb-sd-lora-merged")
    !ct2-transformers-converter --model nllb-sd-lora-merged --quantization int8 --output_dir nllb-sd-ct2-int8 --force \
        --copy_files sentencepiece.bpe.model tokenizer.json tokenizer_config.json special_tokens_map.json
    !zip -qr /kaggle/working/nllb-sd-ct2-int8.zip nllb-sd-ct2-int8 && ls -lh /kaggle/working/*.zip
else:
    print("Fine-tune did not beat the base model -- keep notebook 01's export.")